In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir:Path 
    model_name: str
    target_column: str 
    all_params:dict
    X_train_path : Path 
    X_test_path : Path
    y_train_path : Path
    y_test_path :Path
    THRESHOLD : float
    best_params_path : Path
    
    

In [3]:
from src.constants import *
from src.utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                config_path = Path(CONFIG_FILE_PATH),
                params_path = Path(PARAMS_FILE_PATH),
                schema_path = Path(SCHEMA_FILE_PATH)):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path) 
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
        

    def get_model_trainer(self) -> ModelTrainerConfig:
        config = self.config.model_trainer 
        params = self.params.xgboost_params
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        return ModelTrainerConfig(
            root_dir= Path(config.root_dir),
            model_name=config.model_name,
            target_column = schema.name,
            all_params = params,
            X_train_path = Path(config.X_train_path),
            X_test_path = Path(config.X_test_path),
            y_train_path= Path(config.y_train_path),
            y_test_path= Path(config.y_test_path),
            best_params_path= Path(config.best_params_path),
            THRESHOLD = config.THRESHOLD,
        )

In [9]:
import pandas as pd
import os
from src import logging, CustomException
from xgboost import XGBClassifier
import joblib
import time
from scipy import sparse
from sklearn.metrics import recall_score
import optuna
import yaml
from dataclasses import replace
from src.entity.config_entity import ModelTrainerConfig


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config 
        
        
    def get_best_params(self, n_trials=5):
        """Optuna hyperparameter tuning"""
        logging.info("Find best params")
        params_config = self.config.all_params

        # load data 
        X_train_arr = sparse.load_npz(self.config.X_train_path)
        X_test_arr = sparse.load_npz(self.config.X_test_path)
        # ensure labels are 1d arrays (ravel) so counting and model.fit behave consistently
        y_train = pd.read_csv(self.config.y_train_path).values.ravel()
        y_test = pd.read_csv(self.config.y_test_path).values.ravel()
        
        def objective(trial):
            params = {} 
            for key, cfg in params_config.items():
                if isinstance(cfg, (int, float, str, bool)):
                    params[key] = cfg 
                    continue
                
                p_type = cfg.get("type")
                low = cfg.get("low")
                high = cfg.get("high")
                
                if p_type == "int":
                    params[key] = trial.suggest_int(key,low,high)
                    
                elif p_type == "float":
                    if cfg.get("log", False):
                        params[key] = trial.suggest_float(key, low, high, log=True)
                    else:
                        params[key] = trial.suggest_float(key, low, high)
                
            # handle dynamic scale_pos_weight: compute from y_train counts
            if params.get("scale_pos_weight") == "auto":
                # y_train is a 1-d array of 0/1 labels
                negatives = (y_train == 0).sum()
                positives = (y_train == 1).sum()
                # avoid division by zero; fallback to 1.0 if no positives
                params["scale_pos_weight"] = float(negatives / positives) if positives > 0 else 1.0
                
            # Fixed values 
            params["random_state"] = 42 
            params["n_jobs"] = -1 
            params["eval_metric"] = "logloss"
            
            
            # Evaluate model 
            model = XGBClassifier(**params)       
            model.fit(X_train_arr, y_train)
            proba = model.predict_proba(X_test_arr)[:,1]
            y_pred = (proba >= self.config.THRESHOLD).astype(int)
            return recall_score(y_test ,y_pred, pos_label=1)
        
        
        # run optuna 
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials)     
        
        
        best_params = study.best_params
        logging.info(f"✅ Best parameters found: {best_params}")

        # Merge fixed params and computed scale_pos_weight into final params so the
        # model used for final training matches the one evaluated during Optuna trials.
        final_params = dict(best_params)
        # if original params config asked for auto scale, compute it from y_train
        if params_config.get("scale_pos_weight") == "auto":
            negatives = (y_train == 0).sum()
            positives = (y_train == 1).sum()
            final_params["scale_pos_weight"] = float(negatives / positives) if positives > 0 else 1.0

        # add fixed values used during tuning
        final_params["random_state"] = 42
        final_params["n_jobs"] = -1
        final_params["eval_metric"] = "logloss"

        # save final params (includes best params + fixed values)
        with open(self.config.best_params_path, "w") as f:
            yaml.dump(final_params, f)

        # update config to use final params for training
        self.config = replace(self.config, all_params=final_params)

        return final_params
            
    
    
    def train(self):
        """Train final model using best params"""
        #Initiate Model 
        model = XGBClassifier(**self.config.all_params)
        logging.info(f"🚀 Initializing model: {model.__class__.__name__} with parameters: {model.get_params()}")

        # load data 
        X_train_arr = sparse.load_npz(self.config.X_train_path)
        # ensure labels are 1d array for training
        y_train = pd.read_csv(self.config.y_train_path).values.ravel()

        # Training timer
        start_train = time.time()
        model.fit(X_train_arr, y_train)
        train_time = time.time() - start_train
        print(f"⏱ Training time: {train_time:.2f} seconds")
        
        # save model 
        model_save_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(model, model_save_path)
        logging.info(f"Model {model.__class__.__name__} saved at {model_save_path}")
        
        return model
        
        

In [10]:
config = ConfigurationManager() 
model_trainer_config=config.get_model_trainer()
trainer = ModelTrainer(model_trainer_config) 
best_params = trainer.get_best_params() 
model = trainer.train()

[2025-10-18 23:58:43,213] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-18 23:58:43,218] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-18 23:58:43,222] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-18 23:58:43,223] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-18 23:58:43,224] [INFO] [root:create_directories:39] - created directory at: artifacts/model_trainer
[2025-10-18 23:58:43,225] [INFO] [root:get_best_params:22] - Find best params


[I 2025-10-18 23:58:43,237] A new study created in memory with name: no-name-16538279-d14a-48c7-9287-e86763da529a
[I 2025-10-18 23:58:44,031] Trial 0 finished with value: 0.903485254691689 and parameters: {'n_estimators': 658, 'learning_rate': 0.1850086875433415, 'max_depth': 8, 'subsample': 0.8952322345395503, 'colsample_bytree': 0.9542468530508723, 'min_child_weight': 8, 'gamma': 3.84848426141882, 'reg_alpha': 1.6534996203760994, 'reg_lambda': 1.59003673085419}. Best is trial 0 with value: 0.903485254691689.
[I 2025-10-18 23:58:45,287] Trial 1 finished with value: 0.8900804289544236 and parameters: {'n_estimators': 799, 'learning_rate': 0.13385418680484876, 'max_depth': 6, 'subsample': 0.5162778836085844, 'colsample_bytree': 0.9707858605664366, 'min_child_weight': 5, 'gamma': 3.582455976042791, 'reg_alpha': 2.29840049870817, 'reg_lambda': 1.8386883847816966}. Best is trial 0 with value: 0.903485254691689.
[I 2025-10-18 23:58:46,071] Trial 2 finished with value: 0.900804289544236 and 

[2025-10-18 23:58:48,722] [INFO] [root:get_best_params:80] - ✅ Best parameters found: {'n_estimators': 658, 'learning_rate': 0.1850086875433415, 'max_depth': 8, 'subsample': 0.8952322345395503, 'colsample_bytree': 0.9542468530508723, 'min_child_weight': 8, 'gamma': 3.84848426141882, 'reg_alpha': 1.6534996203760994, 'reg_lambda': 1.59003673085419}
[2025-10-18 23:58:48,726] [INFO] [root:train:111] - 🚀 Initializing model: XGBClassifier with parameters: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.9542468530508723, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': 3.84848426141882, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.1850086875433415, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 

In [11]:
import joblib
import pandas as pd
from scipy import sparse
from sklearn.metrics import recall_score

# Load model
model = joblib.load(r"D:\Programming\ML\End-to-End\End-to-End-TelcoChurn\artifacts\model_trainer\model.joblib")

# Load data
X_test_arr = sparse.load_npz("artifacts/data_transformation/X_test.npz")
y_test = pd.read_csv("artifacts/data_transformation/y_test.csv").values.ravel()

# Use the same threshold as during training
THRESHOLD = 0.3  # or read from YAML config

# Predict
proba = model.predict_proba(X_test_arr)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

# Evaluate
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.4f}")


Recall: 0.9035
